# Laboratorio de Matplotlib con datos del socio formador
**Duración:** 120 minutos · **Entregable:** notebook completado en Canvas

## Objetivo
Construir, personalizar e interpretar gráficas con Matplotlib usando las hojas de inventarios, salidas de almacén y compras del socio formador.

## Cómo trabajar
1. Ejecuta las celdas en orden y lee cada explicación.
2. Cambia los parámetros que se solicitan; no basta con ejecutar el código.
3. Completa las celdas `TODO` y responde las preguntas en Markdown.
4. Guarda el `.ipynb` completado y súbelo a Canvas.



## 1. Preparar el entorno y cargar los datos
Matplotlib dibuja las figuras; pandas lee y transforma las tablas. Si hace falta instalar paquetes, ejecuta `%pip install pandas matplotlib openpyxl`.

Coloca el Excel junto al notebook. Si tiene otro nombre, modifica `ARCHIVO`.

In [ ]:
from pathlib import Path
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

ARCHIVO = Path.cwd().resolve() / "Datos" / "Datos_CLV_2026_Tec_v2.xlsx"
print("Directorio:", Path.cwd())
print("Archivo encontrado:", ARCHIVO.exists())

In [ ]:
if not ARCHIVO.exists():
    raise FileNotFoundError(
        f"No se encontró {ARCHIVO}. Coloca el Excel junto al notebook o corrige ARCHIVO."
    )

libro = pd.ExcelFile(ARCHIVO)
print("Hojas disponibles:", libro.sheet_names)

## 2. Inspeccionar la hoja `Existencias`
Antes de graficar, revisa filas, columnas, tipos de datos y valores faltantes. No supongas que una columna es numérica solo por su nombre.

In [ ]:
existencias = pd.read_excel(ARCHIVO, sheet_name="Existencias_04_09_2026")
existencias.columns = existencias.columns.astype(str).str.strip()

display(existencias.head())
print("Dimensiones (filas, columnas):", existencias.shape)
print("Columnas:", existencias.columns.tolist())
display(existencias.dtypes.to_frame("tipo de dato"))
display(existencias.isna().sum().to_frame("valores faltantes"))

## 3. Resumir antes de graficar
Una barra por marca necesita una cifra por marca. `groupby()` agrupa registros y `sum()` suma las existencias. La gráfica responderá: **¿cuántas unidades están registradas por marca?**

In [ ]:
existencias["existencias"] = pd.to_numeric(existencias["existencias"], errors="coerce")

resumen_marcas = (
    existencias.groupby("marca", as_index=False)["existencias"].sum().sort_values("existencias", ascending=False)
)
display(resumen_marcas)

## 4. Primera gráfica: `Figure` y `Axes`
- `fig` es el lienzo completo.
- `ax` es el área de dibujo que contiene los datos, los ejes y sus etiquetas.
- `figsize=(ancho, alto)` indica el tamaño aproximado en pulgadas.

Usaremos el estilo orientado a objetos recomendado para personalizar gráficas: `ax.bar()`, `ax.set_title()`, etc.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))



ax.bar(resumen_marcas["marca"], resumen_marcas["existencias"])
plt.show()

### 4.1 Añadir título y etiquetas
Una gráfica debe explicar qué muestra. El título describe la pregunta; el eje X identifica las categorías; el eje Y indica la medida y sus unidades.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(resumen_marcas["marca"], resumen_marcas["existencias"], color="darkcyan")
ax.set_title("Existencias registradas por marca", fontsize=15, fontweight="bold")
ax.set_xlabel("Marca", fontsize=11)
ax.set_ylabel("Unidades en existencia", fontsize=11)
plt.xticks(rotation=90, ha="right")  # Inclina las etiquetas y las alinea a la derecha
plt.tight_layout()
plt.show()

**Modifica (5 min):** cambia `figsize` a `(11, 5)`, el título a tamaño 18 y después a 13. Prueba los colores `darkcyan`, `slategray` y `darkorange`. En Markdown, explica qué combinación te parece más legible y por qué.

## 5. Color, borde, transparencia y cuadrícula
`color` cambia el relleno; `edgecolor` el borde; `linewidth` su grosor; `alpha` la transparencia (0 transparente, 1 opaco). La cuadrícula puede facilitar comparar valores. Evita colores decorativos que no tengan una función.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(
    resumen_marcas["marca"], resumen_marcas["existencias"],
    color="cornflowerblue", edgecolor="black", linewidth=0.8, alpha=0.85
)
ax.set_title("Existencias registradas por marca")
ax.set_xlabel("Marca")
ax.set_ylabel("Unidades")
ax.tick_params(axis="x", labelrotation=45)
ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.35)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

**Experimenta:** cambia `alpha` a `0.3` y `1.0`. Cambia `rotation=55` a `0` y `55`. ¿Qué combinación hace que los nombres se lean mejor? ¿Cuándo una cuadrícula demasiado oscura distrae?

## 6. Límites del eje y ticks
Para gráficas de barras, comenzar el eje vertical en cero suele permitir comparaciones más honestas. `set_ylim(bottom=0)` define el límite inferior. Los límites no deben ajustarse para exagerar diferencias ni ocultar información.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(resumen_marcas["marca"], resumen_marcas["existencias"], color="cadetblue")
ax.set_title("Existencias por marca")
ax.set_xlabel("Marca")
ax.set_ylabel("Unidades")
ax.set_ylim(bottom=0)
ax.tick_params(axis="x", rotation=60)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

**Pregunta 2:** ¿por qué podría ser engañoso que el eje vertical de una gráfica de barras comenzara muy por encima de cero?

## 7. Barras horizontales: productos con mayor existencia
`barh()` dibuja barras horizontales. Es útil cuando los nombres de productos son largos. `nlargest(10, ...)` selecciona los registros con valores más altos; luego ordenamos para que la lectura sea clara.

Advertencia: aparecer en el top de existencias no significa, por sí solo, que haya exceso de inventario. Para recomendar compras necesitamos demanda, tiempos de entrega y políticas de inventario.

In [ ]:
productos = existencias.dropna(subset=["nombre_articulo", "existencias"]).copy()
top10 = productos.nlargest(10, "existencias").sort_values("existencias", ascending=True)
display(top10[["nombre_articulo", "existencias"]])

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10["nombre_articulo"].astype(str), top10["existencias"], color="steelblue")
ax.set_title("Productos con mayor cantidad registrada")
ax.set_xlabel("Unidades")
ax.set_ylabel("Producto")
ax.grid(axis="x", linestyle=":", alpha=0.4)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

### 7.1 Destacar una barra
Una barra puede resaltarse si hay una razón clara. En este ejemplo se destaca la de mayor cantidad del conjunto mostrado, no porque sea necesariamente más importante para el negocio.

In [ ]:
colores = ["lightsteelblue"] * len(top10)
if len(colores) > 0:
    colores[-1] = "darkorange"

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10["nombre_articulo"].astype(str), top10["existencias"], color=colores)
ax.set_title("Productos con mayor cantidad registrada")
ax.set_xlabel("Unidades")
ax.set_ylabel("Producto")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

**Modifica (5 min):** cambia `nlargest(10, ...)` por `nlargest(5, ...)`. Cambia el color de resaltado. Explica por qué no se puede recomendar dejar de comprar un producto solo porque aparece en esta gráfica.

## 8. Líneas: evolución temporal de salidas
Las líneas ayudan a visualizar cambios en el tiempo. Convertiremos la fecha a `datetime` y sumaremos las salidas por fecha. Comprueba primero los nombres de las columnas.

In [ ]:
salidas = pd.read_excel(ARCHIVO, sheet_name="Salidas_Almacen")
salidas.columns = salidas.columns.astype(str).str.strip()
print(salidas.columns.tolist())
display(salidas.head())

In [ ]:
salidas["fecha"] = pd.to_datetime(salidas["fecha"], errors="coerce")
salidas["salidas"] = pd.to_numeric(salidas["salidas"], errors="coerce")
salidas_ok = salidas.dropna(subset=["fecha", "salidas"])
serie_salidas = salidas_ok.groupby("fecha")["salidas"].sum().sort_index()
print("Fechas distintas:", serie_salidas.index.nunique())
display(serie_salidas.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(
    serie_salidas.index, serie_salidas.values,
    color="darkgreen", linewidth=2, marker="o", markersize=4,
    linestyle="-", label="Salidas registradas"
)
ax.set_title("Evolución de las salidas de almacén")
ax.set_xlabel("Fecha")
ax.set_ylabel("Unidades de salida")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

### 8.1 Modificar línea y marcadores
Prueba `color="firebrick"`, `linestyle="--"`, `marker="s"` y `markersize=7`. `label` define el texto de la leyenda y `ax.legend()` la muestra.

Si existen muchísimas fechas, los marcadores pueden saturar la gráfica. Si solo existen unas cuantas fechas, no afirmes que hay una tendencia representativa.

## 9. Agregación mensual
Agrupar por mes reduce detalle y cambia la pregunta: el total mensual no muestra cómo se distribuyeron las salidas entre los días del mes.

In [ ]:
salidas_ok = salidas_ok.copy()
salidas_ok["mes"] = salidas_ok["fecha"].dt.to_period("M").dt.to_timestamp()
serie_mensual = salidas_ok.groupby("mes")["salidas"].sum().sort_index()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(serie_mensual.index, serie_mensual.values, color="royalblue", marker="o")
ax.set_title("Total mensual de salidas de almacén")
ax.set_xlabel("Mes")
ax.set_ylabel("Total de unidades")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

**Pregunta 3:** ¿qué información se pierde al agrupar por mes? ¿Qué revisarías antes de afirmar que existe estacionalidad?

## 10. Histograma: distribución de cantidades compradas
Un histograma divide valores numéricos en intervalos (`bins`) y cuenta cuántos registros caen en cada uno. No representa evolución temporal.

In [ ]:
compras = pd.read_excel(ARCHIVO, sheet_name="Histórico_Compras")
compras.columns = compras.columns.astype(str).str.strip()
print(compras.columns.tolist())
display(compras.head())

In [ ]:
compras["unidades"] = pd.to_numeric(compras["unidades"], errors="coerce")
unidades = compras["unidades"].dropna()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(unidades, bins=10, color="mediumpurple", edgecolor="white", alpha=0.85)
ax.set_title("Distribución de unidades por registro de compra")
ax.set_xlabel("Unidades por registro")
ax.set_ylabel("Frecuencia de registros")
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.show()

**Experimenta:** cambia `bins=10` por `bins=5` y `bins=20`. Describe cómo cambia la apariencia y por qué no conviene elegir intervalos para exagerar una conclusión.

## 11. Dispersión: unidades compradas frente a precio
Cada punto representa un registro de compra. El eje X contiene unidades y el Y precio unitario. La relación visual no demuestra causalidad: pueden mezclarse productos, monedas, unidades de medida y condiciones comerciales diferentes.

In [ ]:
compras["precio_unitario"] = pd.to_numeric(compras["precio_unitario"], errors="coerce")
pares = compras.dropna(subset=["unidades", "precio_unitario"])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    pares["unidades"], pares["precio_unitario"],
    color="teal", alpha=0.65, s=45, edgecolor="black", linewidth=0.3
)
ax.set_title("Unidades compradas frente a precio unitario")
ax.set_xlabel("Unidades compradas")
ax.set_ylabel("Precio unitario (verificar moneda y unidad)")
ax.grid(True, alpha=0.25)
plt.show()

**Pregunta 4:** ¿es adecuado comparar todos los productos en una sola dispersión? ¿Qué información adicional necesitarías para interpretar el patrón?

## 12. Diagrama de caja: variabilidad de precios
El boxplot resume mediana y cuartiles y puede mostrar observaciones atípicas. Un valor atípico no necesariamente es un error; puede corresponder a otro producto o condición comercial.

In [ ]:
precios = compras["precio_unitario"].dropna()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.boxplot(precios, patch_artist=True, boxprops={"facecolor": "lightblue"})
ax.set_title("Distribución de precios unitarios registrados")
ax.set_ylabel("Precio unitario")
ax.grid(axis="y", alpha=0.25)
plt.show()

## 13. Subplots: dos vistas en una figura
`plt.subplots(1, 2)` crea dos objetos `Axes`. Cada panel necesita título y etiquetas propios. Una figura compartida no significa que ambas medidas sean comparables directamente si tienen unidades diferentes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(resumen_marcas["marca"], resumen_marcas["existencias"], color="steelblue")
axes[0].set_title("Existencias por marca")
axes[0].set_xlabel("Marca")
axes[0].set_ylabel("Unidades")
axes[0].tick_params(axis="x", rotation=90)
axes[0].grid(axis="y", alpha=0.25)

axes[1].plot(serie_mensual.index, serie_mensual.values, color="darkorange", marker="o")
axes[1].set_title("Salidas mensuales")
axes[1].set_xlabel("Mes")
axes[1].set_ylabel("Unidades de salida")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(True, alpha=0.25)

fig.suptitle("Resumen de inventario y movimientos")
fig.tight_layout()
plt.show()

## 14. Guardar una figura
`savefig()` guarda la figura para utilizarla en un reporte. `dpi` controla la resolución y `bbox_inches="tight"` ayuda a incluir las etiquetas.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(resumen_marcas["marca"], resumen_marcas["existencias"], color="steelblue")
ax.set_title("Existencias por marca")
ax.set_xlabel("Marca")
ax.set_ylabel("Unidades")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig("existencias_por_almacen.png", dpi=200, bbox_inches="tight")
print("Guardada en:", Path("existencias_por_almacen.png").resolve())
plt.show()

# 15. Ejercicios entregables

Completa cada `TODO`, ejecuta el código y responde las preguntas. Se evalúa tanto la visualización como la interpretación.

## Ejercicio A — Personalizar inventario 
Completa el código: modifica tamaño, color, título, etiquetas, cuadrícula y límite inferior del eje Y en cero.

In [ ]:
# TODO: completa y ejecuta
fig, ax = plt.subplots(figsize=(____, ____))
ax.bar(resumen_almacen["nombre_almacen"], resumen_almacen["existencias"], color="____")
ax.set_title("____________________________")
ax.set_xlabel("________________")
ax.set_ylabel("________________")
# Añade cuadrícula horizontal, rotación si hace falta y límite inferior en cero.
plt.show()

**Respuesta A:** ¿qué comparación permite hacer la gráfica y qué limitación tienen los datos?

## Ejercicio B — Top 5 de productos 
Construye barras horizontales con los cinco productos de mayor existencia, ordenadas de menor a mayor en el gráfico.

In [ ]:
# TODO: completa las partes faltantes
top5 = (
    existencias.dropna(subset=["nombre_articulo", "existencias"])
    .nlargest(____, "existencias")
    .sort_values("existencias", ascending=____)
)
fig, ax = plt.subplots(figsize=(____, ____))
ax.barh(top5["nombre_articulo"].astype(str), top5["existencias"], color="________")
ax.set_title("____________________________")
ax.set_xlabel("________________")
ax.set_ylabel("________________")
ax.grid(axis="____", alpha=0.3)
fig.tight_layout()
plt.show()

**Respuesta B:** ¿por qué tener muchas existencias no demuestra que haya exceso ni permite decidir cuánto comprar?

## Ejercicio C — Evolución temporal
Elige `serie_salidas` o `serie_mensual`. Añade color, marcador, estilo de línea, leyenda, título, etiquetas y cuadrícula.

In [ ]:
# TODO: elige una serie válida
serie_elegida = __________

fig, ax = plt.subplots(figsize=(____, ____))
ax.plot(
    serie_elegida.index, serie_elegida.values,
    color="________", marker="____", linestyle="____",
    label="________________"
)
ax.set_title("____________________________")
ax.set_xlabel("________________")
ax.set_ylabel("________________")
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

**Respuesta C:** ¿puedes afirmar que existe tendencia o estacionalidad? Justifica tu respuesta con el periodo y la cantidad de fechas disponibles.

## Ejercicio D — Elige una gráfica
Crea un histograma de unidades, un boxplot de precios o una dispersión de unidades frente a precio. Añade al menos dos personalizaciones y una pregunta de negocio como comentario en el código.

In [ ]:
# TODO: escribe tu solución aquí.
# Pregunta que quiero responder:
# Tipo de gráfica elegido:
# Código:

**Respuesta D:** ¿qué muestra la gráfica y qué no permite concluir?

## Ejercicio E — Reto de dos paneles
Crea una figura con dos gráficas relacionadas. Cada panel debe tener título y etiquetas propios. Añade un título general con `fig.suptitle()`.

In [ ]:
# TODO: completa el reto
fig, axes = plt.subplots(1, 2, figsize=(____, ____))

# Panel izquierdo: crea tu primera gráfica con axes[0].
# Panel derecho: crea tu segunda gráfica con axes[1].

fig.suptitle("____________________________")
fig.tight_layout()
plt.show()

**Respuesta E:** ¿qué aporta ver las dos gráficas juntas? ¿Qué no debe compararse directamente?

## 16. Lista de revisión y reflexión final

**Reflexión final (Markdown):**
1. ¿Qué tipo de gráfica te resultó más útil y por qué?
2. ¿Qué modificación mejoró más la legibilidad?
3. ¿Qué datos adicionales necesitarías para hacer una recomendación empresarial confiable?

